In [0]:
%run ./01_setup_environment

In [0]:
# ========================================
# 09_gold_claims_analytics
# ========================================

from pyspark.sql.functions import *

# Read Silver Layer Tables
claims_df = spark.read.format("delta") \
    .load(f"{silver_path}/claims_clean")

patients_df = spark.read.format("delta") \
    .load(f"{silver_path}/patients_clean")

# Join Claims with Patients
claims_analytics_df = claims_df.join(
    patients_df,
    "patient_id",
    "left"
)

# Aggregate Claims by City
claims_analytics_df = claims_analytics_df.groupBy(
    "city"
).agg(
    sum("claim_amount").alias("total_claim_amount"),
    count("claim_id").alias("total_claims")
)

# Write to Gold Layer
claims_analytics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/claims_analytics")

print("Claims Analytics Completed")